# 用 SPX-CD 学会做决策：全双工交互入门

当助手正在讲话，用户说“停一下，我想先问第一步”，程序该做什么？我们用这个场景学习如何调用决策模型、读取候选概率，再把选择接到程序动作上。

本文选用 **Surd AI 的 Simplex CD（SPX-CD）**，因为它近期提供免费体验，方便第一次接触决策模型的读者直接动手。2026 年 10 月编写本文时，官网标注为免费公测，当前体验政策可在[官网](https://surdai.com/en)查看。

开始前，到[官网注册账户](https://surdai.com/en/register)，登录后在 [Access tokens](https://surdai.com/en/platform/keys) 页面创建 API key。保存好密钥，后面的初始化代码会读取它。[平台文档](https://surdai.com/en/platform/docs)提供完整接口说明。

按顺序阅读基础部分即可完成第一次调用。批量测试和扩展练习按需开启；已有 Jev 账户的读者可以查看末尾的官方 SDK 迁移示例。

项目定位和文献见 [README](README.md)。完成本篇后，可以继续 [EasyTurn 实验](Jev_EasyTurn_text_experiment.ipynb)，用 zero-shot Jev 文本基线观察话轮测试集的语义偏好。

## 1. 准备环境

In [ ]:
# 第一次使用时取消下一行的注释，安装到当前 notebook 的 Python 环境。
# %pip install -q git+https://github.com/SPX-AI968/spx-sdk.git numpy pandas matplotlib

上面的安装命令使用 SurdAI 文档链接的 Python SDK。后面的 NumPy、pandas 和 Matplotlib 用于计算指标与绘图。

接下来创建连接，API key 可以从环境变量读取，也可以在隐藏输入框中填写。

In [9]:
import os
from getpass import getpass

from simplex_ai import Choice, SimplexClient

BASE_URL = "https://api.surdai.com"
API_KEY = os.getenv("SIMPLEX_API_TOKEN") or globals().get("API_KEY") or getpass("SurdAI token: ")
client = SimplexClient(api_key=API_KEY, base_url=BASE_URL)
TEXT_MODEL = "spx-cd-flash"

`client` 是后续调用模型的入口。`base_url` 指向 SurdAI 服务，`api_key` 用于鉴权。当前教程使用的 SDK 包名是 `simplex-ai-sdk`，Python 中从 `simplex_ai` 导入。

[SDK 安装与客户端说明](https://github.com/SPX-AI968/spx-sdk#install)

`TEXT_MODEL` 指定基础文本示例使用的模型，默认是 `spx-cd-flash`。使用 Surd 的 key 时，把它改为 `typesafe-ai/jev`，即可用同一套调用方式体验 Jev。

## 2. 选择下一步动作

In [10]:
state = {
    "assistant_speaking": True,
    "assistant_text": "下面解释流式推理的第三个步骤……",
    "user_text": "停一下，我想先问第一步。",
    "transcript_status": "final",
    "pause_ms": 0,
}

`state` 是我们传给模型的当前情景，可以用普通 Python 字典组织。这里助手正在讲解，用户要求暂停并回到第一步。

其中 `assistant_speaking` 表示助手是否正在播报；两个文本字段记录双方正在说什么。`transcript_status="final"` 表示收到的文本片段已经稳定，`pause_ms` 记录用户停顿的毫秒数。这些字段由应用定义，帮助模型理解交互状态。

下一步把程序允许执行的动作列出来，并写清选择规则。

In [11]:
LABELS = {
    "stop_speaking": "立即停止当前播报，把话轮交给用户",
    "continue_speaking": "继续当前播报，用户未要求打断",
    "wait_for_user": "保持静默，等待用户完成表达",
    "respond": "用户已交回话轮，开始回应",
}
POLICY = (
    "根据助手播报状态和用户文本，选择下一步动作。"
    "助手播报中，用户要求停止、暂停或抢回话轮：stop_speaking；"
    "助手播报中，用户只作附和或要求继续：continue_speaking；"
    "助手未播报，用户仍在表达或明确表示准备补充：wait_for_user；"
    "助手未播报，用户说完并交回话轮：respond。"
    "根据用户内容和上下文判断话轮是否结束。"
)

`LABELS` 的键是程序使用的动作标识，值是给模型看的动作说明。`POLICY` 描述如何根据情景选择动作。

这就是决策模型（Decision model）的基本用法：提供当前情景和允许的结果，让模型选择一个结果，并给出相应概率。在全双工系统中，这个结果可以控制继续播报、让出话轮或开始回应。

In [12]:
decision_result = client.system_one(
    model=TEXT_MODEL,
    state=state,
    questions={
        "action": Choice(instructions=POLICY, criteria=LABELS),
    },
)
answer = decision_result.answers["action"]
print("模型选择：", answer.choice)

模型选择： stop_speaking


`system_one()` 发出一次请求。`model` 指定使用的模型，`state` 提供情景，`questions` 放要回答的问题。

我们把问题命名为 `action`。`Choice` 表示从候选中选一个结果；`instructions` 接收选择规则，`criteria` 接收候选动作。读取结果时，使用同一个问题名取出 `answers["action"]`，再通过 `choice` 得到动作标识。

这个情景的预期动作是 `stop_speaking`：先暂停当前播报，让用户完成提问。

In [13]:
probabilities = answer.probabilities
if probabilities is None:
    raise ValueError("本次响应缺少候选概率，请检查完整响应。")

ranked_actions = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)
for action, probability in ranked_actions:
    print(f"{action}: {probability:.3f}")

best_action, best_probability = ranked_actions[0]
second_action, second_probability = ranked_actions[1]
probability_margin = best_probability - second_probability
print(f"最倾向的动作：{best_action} · 概率={best_probability:.3f}")
print(f"第二个动作：{second_action} · 概率={second_probability:.3f}")
print(f"前两名概率差：{probability_margin:.3f}")

stop_speaking: 0.948
respond: 0.036
wait_for_user: 0.011
continue_speaking: 0.006
最倾向的动作：stop_speaking · 概率=0.948
第二个动作：respond · 概率=0.036
前两名概率差：0.912


`Choice` 已经返回每个候选的概率。上面按概率排序，找出模型最倾向的两个动作，并计算它们的概率差。

最高概率描述模型对这个动作的倾向，前两名的差距帮助我们观察选择是否集中。两个动作接近时，可以结合下一段用户文本继续判断。

接下来根据选中动作的概率，给程序加一个可调整的执行阈值。

In [14]:
action_probability = probabilities[answer.choice]
execution_threshold = 0.8

# 先检查选中动作的概率，再进入对应的执行分支。
if action_probability < execution_threshold:
    print("运行时动作：继续收集用户输入，更新情景后再判断。")
elif answer.choice == "stop_speaking":
    print("运行时动作：暂停当前播报，继续监听用户。")
elif answer.choice == "continue_speaking":
    print("运行时动作：继续播报当前内容。")
elif answer.choice == "wait_for_user":
    print("运行时动作：保持静默，等待下一段用户输入。")
else:
    print("运行时动作：开始生成并播报回应。")

运行时动作：暂停当前播报，继续监听用户。


`action_probability` 读取实际选中动作的概率，`execution_threshold` 控制何时进入执行分支。这里先用 0.8 做练习，后面可以结合标注样本和误打断的代价调整。

接入实际系统时，把 `print()` 换成相应的播放、监听或生成操作。概率帮助决定何时行动，动作标识决定执行哪一步。

In [15]:
server_inference_ms = decision_result.model_dump().get("inference_ms")
if server_inference_ms is not None:
    print(f"服务端推理耗时：{server_inference_ms:.1f} ms")
else:
    print("本次响应未返回服务端推理耗时。")

服务端推理耗时：357.1 ms


`inference_ms` 是 API 返回的服务端推理耗时，单位是毫秒。通过 `model_dump()` 可以读取响应中的扩展字段。后面的比较继续使用这个字段，缺失时保留为空。

## 3. 另外三种输出形式

### 判断一个条件

In [16]:
from simplex_ai import Noul

interrupt_result = client.system_one(
    model=TEXT_MODEL,
    state=state,
    questions={
        "interrupt": Noul(instructions="用户是否明确要求停止当前播报？"),
    },
)
interrupt_probability = interrupt_result.answers["interrupt"].noul
print(f"应停止当前播报的概率：{interrupt_probability:.3f}")

应停止当前播报的概率：0.965


`Noul` 回答一个是非问题，返回“是”的概率，范围是 0 到 1。这里问的是用户是否要求停止当前播报。

当应用只关心一个条件是否成立时，这种形式很直接。比如程序可以根据这个概率决定是否触发暂停。

### 评估程度

In [17]:
from simplex_ai import Score

strength_result = client.system_one(
    model=TEXT_MODEL,
    state=state,
    questions={
        "strength": Score(
            instructions="评估用户要求打断助手的意图强度。",
            criteria=["没有打断意图", "打断意图含糊", "明确要求停止或抢回话轮"],
        ),
    },
)
strength = strength_result.answers["strength"].score
print(f"打断意图强度：{strength:.2f} / 2")

strength_answer = strength_result.answers["strength"]
if strength_answer.probabilities is not None:
    for level, probability in strength_answer.probabilities.items():
        print(f"等级 {level} 的概率：{probability:.3f}")

打断意图强度：1.99 / 2
等级 0 的概率：0.002
等级 1 的概率：0.004
等级 2 的概率：0.994


`Score` 使用有序候选。上面三个等级的索引依次是 0、1、2，返回值是这些索引的概率加权平均，因此可以是小数。

越接近 2，表示模型越倾向于“明确要求打断”。这种形式适合用一个连续数值表达程度。

同时显示的等级概率，可以帮助理解这个平均分由哪些等级构成。

### 提取多个意图

In [18]:
from simplex_ai import MultiChoice

intent_result = client.system_one(
    model="spx-cd-flash",
    state=state,
    questions={
        "intents": MultiChoice(
            instructions="选出用户明确表达的全部交互意图。",
            criteria={
                "pause": "暂停当前播报",
                "ask_previous": "询问之前的步骤",
                "continue": "要求继续播报",
            },
            min_choices=0,
            max_choices=2,
        ),
    },
)
print("用户表达的意图：", intent_result.answers["intents"].choices)

intent_answer = intent_result.answers["intents"]
for selected_intent, step in zip(intent_answer.choices, intent_answer.steps or []):
    step_probability = step.get("probabilities", {}).get(selected_intent)
    print(f"{selected_intent}: 本步选择概率={step_probability}")

用户表达的意图： ['ask_previous', 'pause']
ask_previous: 本步选择概率=0.592414277158252
pause: 本步选择概率=0.8806577965715903


`MultiChoice` 可以选多个候选。这个例子同时包含“暂停播报”和“询问之前步骤”，预期选出 `pause` 和 `ask_previous`。

`min_choices` 和 `max_choices` 限定可选择的数量。至此，我们已经练习了四种问题形式：选一个动作、判断一个条件、评估程度、提取多个意图。

[SDK 的四种问题类型](https://github.com/SPX-AI968/spx-sdk#four-question-types)

多选逐步构建结果，每一步的概率都结合此前已经选出的意图。上面打印各个选中意图在对应步骤中的概率。

这一节固定使用 SPX-CD 测试多选。Jev 用户可以练习前面的 Choice、Noul 和 Score，继续使用同一套文本情景。

## 4. 理解概率和训练目标

In [19]:
# 用一组演示数值观察概率与正确率的关系。
reported_probabilities = [0.8] * 10
correct_decisions = [1] * 8 + [0] * 2

mean_probability = sum(reported_probabilities) / len(reported_probabilities)
accuracy = sum(correct_decisions) / len(correct_decisions)
print(f"平均概率：{mean_probability:.0%}")
print(f"实际正确率：{accuracy:.0%}")

平均概率：80%
实际正确率：80%


这组演示中，10 次决策都给出 80% 的概率，其中 8 次正确，概率与正确率相匹配。这种匹配称为**校准（calibration）**。它关注一组决策的表现：当模型经常给出某个概率时，实际正确的比例应接近这个概率。

校准好的概率可以帮助应用做选择。例如，明确的打断请求可以立即触发暂停，含糊的输入则可以等下一段文本再判断。阈值要结合误打断和漏打断的代价，以及标注样本上的结果来选择。

TypeSafe 将用于 Jev 的训练方法称为 **RLCD（Reinforcement Learning for Calibrated Decisions）**，把校准的决策作为强化学习的优化目标。对使用者而言，重要的是同时观察模型是否选对，以及它给出的概率是否与实际表现相符。接下来用标注样本检查 SPX-CD 和 Jev 的这两方面表现。

[TypeSafe 对 RLCD 与 Jev 的介绍](https://typesafe.ai/blog/introducing-system-one-models-and-jev)

## 5. 准备全双工练习样本

In [ ]:
examples = [
    {
        "id": "stop01",
        "gold": "stop_speaking",
        "rationale": "明确要求暂停播报",
        "ood": False,
        "state": {
            "assistant_speaking": True,
            "transcript_status": "final",
            "pause_ms": 0,
            "context": "助手正在讲解流式推理。",
            "user_text": "停一下，我想先问第一步。",
        },
    },
    {
        "id": "continue01",
        "gold": "continue_speaking",
        "rationale": "附和并明确要求继续",
        "ood": False,
        "state": {
            "assistant_speaking": True,
            "transcript_status": "final",
            "pause_ms": 0,
            "context": "助手正在讲解流式推理。",
            "user_text": "嗯嗯，好。",
        },
    },
    {
        "id": "wait01",
        "gold": "wait_for_user",
        "rationale": "条件从句尚未完成",
        "ood": False,
        "state": {
            "assistant_speaking": False,
            "transcript_status": "partial",
            "pause_ms": 300,
            "context": "助手正在静默听取用户输入。",
            "user_text": "我的问题是，如果模型先",
        },
    },
]

现在每条样本除了 `state`，还带有 `id`、`gold` 和 `rationale`。`id` 标识样本，`gold` 是按前面的交互规则标注的正确动作，`rationale` 说明标注理由。`ood` 用于标记后面的缺失状态案例。

调用模型时只传入 `state`；其余字段留在本地，用来检查回答。这一组覆盖明确打断、要求继续和表达未完成，下面补充更多对照。

In [21]:
examples.extend(
    [
        {
            "id": "respond01",
            "gold": "respond",
            "rationale": "完整请求并明确交回话轮",
            "ood": False,
            "state": {
                "assistant_speaking": False,
                "transcript_status": "final",
                "pause_ms": 800,
                "context": "助手正在静默听取用户输入。",
                "user_text": "请解释什么是流式解码。",
            },
        },
        {
            "id": "stop02",
            "gold": "stop_speaking",
            "rationale": "明确抢回话轮",
            "ood": False,
            "state": {
                "assistant_speaking": True,
                "transcript_status": "final",
                "pause_ms": 0,
                "context": "助手正在讲解流式推理。",
                "user_text": "不用继续讲了，先听我说。",
            },
        },
        {
            "id": "continue02",
            "gold": "continue_speaking",
            "rationale": "确认理解并要求继续",
            "ood": False,
            "state": {
                "assistant_speaking": True,
                "transcript_status": "final",
                "pause_ms": 0,
                "context": "助手正在讲解流式推理。",
                "user_text": "对，我明白这个概念。接着讲。",
            },
        },
    ]
)

这一组包含完整提问，以及播报过程中的打断与附和。读样本时先判断助手是否正在播报，再结合用户内容选择动作。

In [22]:
examples.extend(
    [
        {
            "id": "wait02",
            "gold": "wait_for_user",
            "rationale": "句子完整但明确保留话轮",
            "ood": False,
            "state": {
                "assistant_speaking": False,
                "transcript_status": "final",
                "pause_ms": 800,
                "context": "助手正在静默听取用户输入。",
                "user_text": "先别回答，我还要补充一个条件。",
            },
        },
        {
            "id": "respond02",
            "gold": "respond",
            "rationale": "已交回话轮，应回应第一步",
            "ood": False,
            "state": {
                "assistant_speaking": False,
                "transcript_status": "final",
                "pause_ms": 800,
                "context": "助手正在静默听取用户输入。",
                "user_text": "我的问题就这些。",
            },
        },
        {
            "id": "stop03",
            "gold": "stop_speaking",
            "rationale": "播报中应先停止，再由后续状态决定回应",
            "ood": False,
            "state": {
                "assistant_speaking": True,
                "transcript_status": "final",
                "pause_ms": 0,
                "context": "助手正在讲解流式推理。",
                "user_text": "等一下，先解释第一个步骤。",
            },
        },
    ]
)

“先别回答，我还要补充”虽然是完整句子，用户仍保留话轮；“我的问题就这些”则交回了话轮。两者在动作选择上分别对应等待与回应。

In [23]:
examples.extend(
    [
        {
            "id": "continue03",
            "gold": "continue_speaking",
            "rationale": "当前播报中的简短附和",
            "ood": False,
            "state": {
                "assistant_speaking": True,
                "transcript_status": "final",
                "pause_ms": 200,
                "context": "助手正在讲解流式推理。",
                "user_text": "好的。",
            },
        },
        {
            "id": "wait03",
            "gold": "wait_for_user",
            "rationale": "长停顿但内容未完成",
            "ood": False,
            "state": {
                "assistant_speaking": False,
                "transcript_status": "partial",
                "pause_ms": 1200,
                "context": "助手正在静默听取用户输入。",
                "user_text": "我要比较两个方案，第一个是……",
            },
        },
        {
            "id": "respond03",
            "gold": "respond",
            "rationale": "此前助手明确询问是否开始下一节，用户确认后交回话轮",
            "ood": False,
            "state": {
                "assistant_speaking": False,
                "transcript_status": "final",
                "pause_ms": 800,
                "context": "助手刚询问：是否现在开始下一节？用户回答后已交回话轮。",
                "user_text": "好的。",
            },
        },
    ]
)

现在共有 12 条样本，四类各 3 条。“好的”在助手播报中是附和，在助手询问是否开始下一节后则是确认。对话前文决定了同一句文本的作用。

In [24]:
for sample in [examples[9], examples[11]]:
    print(sample["state"]["context"])
    print(f"用户：{sample['state']['user_text']}")
    print(f"正确动作：{sample['gold']} · {sample['rationale']}\n")

助手正在讲解流式推理。
用户：好的。
正确动作：continue_speaking · 当前播报中的简短附和

助手刚询问：是否现在开始下一节？用户回答后已交回话轮。
用户：好的。
正确动作：respond · 此前助手明确询问是否开始下一节，用户确认后交回话轮



这两条对照可以帮助检查模型是否使用了运行状态和前文。下面配置一次批量练习。

In [27]:
MODELS = ["spx-cd-flash", "spx-cd-pro", "typesafe-ai/jev"]
N_SAMPLES = 12
RUN_PAIRED = False

print(f"开启配对测试后将发送 {N_SAMPLES * len(MODELS)} 次请求。")

开启配对测试后将发送 36 次请求。


`MODELS` 指定要比较的三个模型。Jev 可通过同一个 key 调用，模型标识为 `typesafe-ai/jev`。`N_SAMPLES` 控制使用多少条样本。

`RUN_PAIRED` 默认关闭。准备实测时改为 `True`，再运行下面的采集代码；设成只包含一个模型的列表，可以先做较小规模的练习。

In [28]:
from simplex_ai import SimplexError

paired_runs = []
failures = []
if RUN_PAIRED:
    for index, sample in enumerate(examples[:N_SAMPLES]):
        # 轮换调用顺序，让模型交替先调用。
        offset = index % len(MODELS)
        model_order = MODELS[offset:] + MODELS[:offset]
        for model in model_order:
            try:
                result = client.system_one(
                    model=model,
                    state=sample["state"],
                    questions={"action": Choice(instructions=POLICY, criteria=LABELS)},
                )
                paired_runs.append(
                    {
                        "id": sample["id"],
                        "model": model,
                        "gold": sample["gold"],
                        "state": sample["state"],
                        "response": result.model_dump(),
                    }
                )
            except SimplexError as exc:
                failures.append({"id": sample["id"], "model": model, "error": str(exc)})
                print(f"{model}/{sample['id']}: {exc}")
print(f"成功采集 {len(paired_runs)} 条响应。")

成功采集 36 条响应。


采集循环对每条样本分别调用各个模型，把完整响应保存在 `paired_runs` 中。失败项放入 `failures`，方便之后定位问题。

下一步从响应中提取动作、概率和服务端耗时，整理成用于计算的表格。

In [ ]:
import numpy as np
import pandas as pd

records = []
for run in paired_runs:
    response = run["response"]
    action_answer = response["answers"]["action"]
    probabilities = action_answer.get("probabilities")
    if probabilities is None:
        continue

    probability_vector = np.array([probabilities[key] for key in LABELS])
    if not (
        np.isfinite(probability_vector).all()
        and (probability_vector >= 0).all()
        and np.isclose(probability_vector.sum(), 1, atol=1e-5)
    ):
        raise ValueError(f"{run['model']}/{run['id']}: 候选概率无效。")

    records.append(
        {
            "id": run["id"],
            "model": run["model"],
            "gold": run["gold"],
            "choice": action_answer["choice"],
            "probs": probability_vector,
            "inference_ms": response.get("inference_ms"),
        }
    )
paired = pd.DataFrame(records)
print(f"可用于计算的预测：{len(paired)} 条。")

`records` 中的一行对应一个模型对一条样本的回答。`choice` 用于判断动作是否正确，`probs` 按 `LABELS` 的顺序保存所有候选概率，`inference_ms` 保存服务端推理耗时。

接下来计算三项质量指标，并汇总服务端耗时。

In [ ]:
def calibration_metrics(frame):
    probabilities = np.stack(frame.probs)
    gold_indices = np.array([list(LABELS).index(label) for label in frame.gold])
    gold_probabilities = probabilities[np.arange(len(frame)), gold_indices]

    # 每行只在正确动作的位置记为 1，用于比较整组候选概率。
    targets = np.eye(len(LABELS))[gold_indices]
    valid_times = frame.inference_ms.dropna()
    return {
        "n": len(frame),
        "accuracy": (frame.choice == frame.gold).mean(),
        "nll": -np.log(np.clip(gold_probabilities, 1e-12, 1)).mean(),
        "brier": np.square(probabilities - targets).sum(axis=1).mean(),
        "inference_ms": valid_times.median() if len(valid_times) else None,
        "timed_n": len(valid_times),
    }

上面的函数把一组回答汇总为：

- **Accuracy：** 实际动作选对的比例，越高越好。
- **NLL：** 对正确动作概率取负对数后求平均。模型把正确答案的概率压得很低时，惩罚会很大，越低越好。
- **Brier：** 把候选概率与正确动作的 0/1 标记逐项比较，平方误差求和后再平均，越低越好。

NLL 和 Brier 同时考虑选择结果与概率质量。`inference_ms` 汇总服务端耗时的中位数，`timed_n` 记录提供该字段的样本数。

In [ ]:
summaries = {}
matched = pd.DataFrame()
if not paired.empty:
    # 每个模型都完成的样本用于配对比较。
    completed = paired.groupby("id").model.nunique()
    shared_ids = completed[completed == len(MODELS)].index
    matched = paired[paired.id.isin(shared_ids)]
    for model, frame in matched.groupby("model", sort=False):
        metrics = calibration_metrics(frame)
        summaries[model] = metrics
        print(
            f"{model} · n={metrics['n']} · Accuracy={metrics['accuracy']:.1%}"
            f" · NLL={metrics['nll']:.3f} · Brier={metrics['brier']:.3f}"
        )
        if metrics["inference_ms"] is not None:
            print(
                f"  服务端推理中位数：{metrics['inference_ms']:.1f} ms"
                f" · 有耗时字段的样本：{metrics['timed_n']}"
            )
        else:
            print("  服务端推理耗时：暂无返回值。")
if not summaries:
    print("开启配对测试并完成采集后，这里显示比较结果。")

先看 Accuracy，判断动作是否选对；再看 NLL 和 Brier，判断模型如何给正确动作分配概率；最后看服务端推理耗时。

下一幅图把这三个观察角度并排展示。

In [ ]:
import matplotlib.pyplot as plt

COLORS = {
    "spx-cd-flash": "#296bc3",
    "spx-cd-pro": "#17a589",
    "typesafe-ai/jev": "#b36ad1",
}
plot_specs = [
    ("accuracy", "Accuracy ↑"),
    ("brier", "Brier ↓"),
    ("inference_ms", "Server inference median (ms) ↓"),
]

In [ ]:
if summaries:
    metric_frame = pd.DataFrame(summaries).T.reindex(MODELS)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    for ax, (metric, title) in zip(axes, plot_specs):
        values = metric_frame[metric].dropna()
        names = [model.replace("spx-cd-", "").replace("typesafe-ai/", "") for model in values.index]
        ax.bar(names, values, color=[COLORS[model] for model in values.index])
        ax.set_title(title)
        for index, value in enumerate(values):
            text = f"{value:.1f}" if metric == "inference_ms" else f"{value:.3f}"
            ax.annotate(
                text, (index, value), xytext=(0, 4), textcoords="offset points", ha="center"
            )
        if values.empty:
            ax.text(0.5, 0.5, "No server timing returned", ha="center", transform=ax.transAxes)
        ax.margins(y=0.2)
    fig.tight_layout()
    plt.show()

图中 Accuracy 越高越好，Brier 越低越好。服务端耗时反映这批输入对应的计算时间。若想定位某个动作的问题，可以逐条检查它的选择。

In [38]:
if summaries:
    error_count = (matched.choice != matched.gold).sum()
    if error_count == 0:
        print(f"没有预测错误：{len(matched)} 条配对预测全部正确。")
    else:
        print(f"共有 {error_count} 条错误预测，相关样本如下：")
        for sample in examples[:N_SAMPLES]:
            rows = matched[matched.id == sample["id"]]
            if (rows.choice != rows.gold).any():
                print(f"{sample['id']}: {sample['state']['user_text']}")
                print(f"预期：{sample['gold']} · {sample['rationale']}")
                for row in rows.itertuples():
                    gold_index = list(LABELS).index(row.gold)
                    print(
                        f"  {row.model}: {row.choice}"
                        f" · P(正确动作)={row.probs[gold_index]:.3f}"
                    )
else:
    print("尚无配对比较结果，请先完成前面的采集与汇总。")

没有预测错误：36 条配对预测全部正确。


这些错误案例把汇总指标重新连接到具体输入。调整情景描述或动作规则时，可以先观察同一组输入上的变化，再扩展样本。

## 6. 扩展练习：查看模型与尝试额外功能

In [39]:
catalog = client.models()
for item in catalog["data"]:
    print(
        f"{item['id']} · questions={item.get('question_types')}"
        f" · provider={item.get('provider', '未披露')}"
    )

spx-cd-flash · questions=['choice', 'noul', 'score', 'multi_choice'] · provider=未披露
spx-cd-pro · questions=['choice', 'noul', 'score', 'multi_choice'] · provider=未披露
spx-cd · questions=['choice', 'noul', 'score', 'multi_choice'] · provider=未披露
spx-cd-auto · questions=['choice', 'noul', 'score', 'multi_choice'] · provider=未披露
typesafe-ai/jev · questions=['choice', 'noul', 'score'] · provider=vercel-ai-gateway


`client.models()` 返回当前模型目录。`question_types` 表示可使用的问题形式，`provider` 表示目录报告的提供方或网关。

Jev 在这个 SDK 中支持文本 Choice、Noul 和 Score。SPX-CD 可继续用于多选与下面的 reasoning 练习。[SDK 模型说明](https://github.com/SPX-AI968/spx-sdk#models)

In [40]:
reason_state = {
    "assistant_speaking": False,
    "user_text": "先别回答，我还要补充一个条件。",
    "transcript_status": "final",
    "pause_ms": 800,
}
RUN_REASONING = False
reasoning_runs = []

In [41]:
if RUN_REASONING:
    for enabled, budget in [(False, 512), (True, 128), (True, 512)]:
        result = client.system_one(
            model="spx-cd-pro",
            state=reason_state,
            questions={"action": Choice(instructions=POLICY, criteria=LABELS)},
            reasoning=enabled,
            reasoning_max_tokens=budget,
        )
        setting = f"budget_{budget}" if enabled else "off"
        inference_ms = result.model_dump().get("inference_ms")
        reasoning_runs.append({"setting": setting, "response": result.model_dump()})
        print(f"{setting}: {result.answers['action'].choice} · inference_ms={inference_ms}")

off: wait_for_user · inference_ms=307.7091407030821


/var/folders/zk/3lkxlcv90wz6snbk2x0c6syw0000gn/T/ipykernel_56201/1499889247.py:3: UserWarning: Reasoning is experimental and not recommended for normal use.
  result = client.system_one(


budget_128: wait_for_user · inference_ms=1550.6580285727978
budget_512: wait_for_user · inference_ms=1542.0229081064463


`reasoning=True` 开启 SDK 的实验性推理选项，`reasoning_max_tokens` 控制预算。把 `RUN_REASONING` 改为 `True` 后，会比较关闭、128 和 512 三档，共发送 3 次请求。

这条输入的预期动作是 `wait_for_user`。观察选择是否随预算变化，以及服务端耗时怎样变化。[SDK reasoning 说明](https://github.com/SPX-AI968/spx-sdk#errors-and-retries)

In [42]:
missing_examples = [
    {
        "id": "missing01",
        "gold": None,
        "rationale": "缺少播报状态和前文，不能确定动作",
        "ood": True,
        "state": {"user_text": "好的。"},
    },
    {
        "id": "missing02",
        "gold": None,
        "rationale": "缺少播报状态，停止播报或保持等待均可能",
        "ood": True,
        "state": {"user_text": "等一下。"},
    },
]

In [44]:
RUN_MISSING_STATE = False
missing_runs = []
if RUN_MISSING_STATE:
    for sample in missing_examples:
        result = client.system_one(
            model="spx-cd-flash",
            state=sample["state"],
            questions={"action": Choice(instructions=POLICY, criteria=LABELS)},
        )
        response_answer = result.answers["action"]
        missing_runs.append({"id": sample["id"], "response": result.model_dump()})
        print(sample["state"]["user_text"], response_answer.choice, response_answer.probabilities)

好的。 respond {'stop_speaking': 0.09682700745765291, 'continue_speaking': 0.09988989948102867, 'wait_for_user': 0.04023308600102793, 'respond': 0.7630500070602905}
等一下。 stop_speaking {'stop_speaking': 0.8459535735903181, 'continue_speaking': 0.016627007981165197, 'wait_for_user': 0.11448736648687038, 'respond': 0.022932051941646367}


这两条案例只保留“好的”和“等一下”。开启 `RUN_MISSING_STATE` 后，比较它们与前面带上下文的案例，看看缺少运行状态时，动作选择和概率会怎样变化。

In [45]:
from simplex_ai import AsyncSimplexClient

RUN_ASYNC = False
async_response = None
if RUN_ASYNC:
    async with AsyncSimplexClient(api_key=API_KEY, base_url=BASE_URL) as async_client:
        async_result = await async_client.system_one(
            model="spx-cd-flash",
            state=state,
            questions={"interrupt": Noul(instructions="用户是否明确要求停止当前播报？")},
        )
    async_response = async_result.model_dump()
    print(f"异步调用结果：{async_result.answers['interrupt'].noul:.3f}")

异步调用结果：0.970


`AsyncSimplexClient` 使用 `await` 等待结果，适合接到已有的异步应用中。这里仍然是同一条文本打断请求；把 `RUN_ASYNC` 改为 `True` 就会发送一次请求。

In [ ]:
client.close()

到这里，你已经会用情景、候选动作和选择规则调用 SPX-CD，读取结构化结果与概率，并用标注样本检查决策质量。

下一篇 [EasyTurn 实验](Jev_EasyTurn_text_experiment.ipynb)复用这种调用方式，检查通用决策模型仅凭 ASR 文本能取得怎样的成绩。
已完成的实验为 EasyTurn 测试集较强的语义偏好提供了佐证，并引出后续标注与合成应结合声学和对话接续信息的问题。
结果与讨论见 [实验报告](EXPERIMENT_REPORT.md)。

## 7. 已有 Jev 账户：迁移到官方 SDK

下面复用前面定义的对话与动作规则，通过 Jev 官方服务完成同一个动作选择。

In [ ]:
# 使用这个可选示例前，先安装官方 SDK。
# %pip install -q typesafe-sdk

In [ ]:
RUN_NATIVE_JEV = False

if RUN_NATIVE_JEV:
    import os
    from getpass import getpass

    from typesafe_sdk import Choice as JevChoice
    from typesafe_sdk import TypeSafeClient

    jev_api_key = os.getenv("TYPESAFE_API_KEY") or getpass("TypeSafe token: ")
    with TypeSafeClient(api_key=jev_api_key) as jev_client:
        jev_result = jev_client.system_one(
            state=state,
            questions={
                "action": JevChoice(instructions=POLICY, criteria=LABELS),
            },
        )
    jev_answer = jev_result.choices["action"]
    print("Jev 选择：", jev_answer.choice)
    print("候选概率：", jev_answer.probabilities)

`RUN_NATIVE_JEV` 默认关闭。安装后改为 `True`，使用 [TypeSafe 控制台](https://console.typesafe.ai/)创建的 key，或提前设置环境变量 `TYPESAFE_API_KEY`。

迁移的主要改动是导入 `typesafe_sdk`、创建 `TypeSafeClient`，以及从 `response.choices["action"]` 读取结果。官方客户端默认调用 `jev-latest`；前面的 `state`、`LABELS` 和 `POLICY` 可以继续使用。

同理，使用官方的 `Noul` 和 `Score` 定义问题后，通过 `response.nouls[问题名]` 和 `response.scores[问题名]` 读取结果。批量样本可以沿用前面的数据，只需替换调用与结果读取部分。

[官方 Python SDK](https://docs.typesafe.ai/sdk/python) · [Quick start](https://docs.typesafe.ai/introduction/quickstart)